In [1]:
# Imports
import sys
sys.path.append("Program")

from backtest import *
import concurrent.futures
import datetime as dt
from functools import partial
from fundamentals import *
from helper_functions import modify_current_date, get_df, get_infix
import matplotlib.pyplot as plt
import multiprocessing
import numpy as np
import pandas as pd
from plot import *
from technicals import *
from tqdm import tqdm

# Start of the program
start = dt.datetime.now()

# Variables
all_stocks = False
period_short = 60
period_long = 252
RS = 90
factors = [0.1, 0.55, 0.35]
backtest = False

# Index
index_name = "^GSPC"
index_dict = {"^HSI": "Hang Seng Index", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite"}

# Get the infix
infix = get_infix("^GSPC", index_dict, all_stocks) 

# Modify the current date
current_date = modify_current_date(start, index_name)

In [2]:
# Index
indices = ["^HSI", "^GSPC", "^IXIC", "QLD", "TQQQ"]
index_dict = {"^HSI": "Hang Seng Index", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite", "QLD": "ProShares Ultra QQQ", "TQQQ": "ProShares UltraPro QQQ"}

# Momentum ETFs
etfs_dict = {
    "MTUM": "iShares MSCI USA Momentum Factor ETF",
    "IMTM": "iShares MSCI Intl Momentum Factor ETF",
    "SMLF": "iShares U.S. Small‑Cap Equity Factor ETF",
    "IUSV": "iShares S&P U.S. Value ETF",
    "IJJ": "iShares S&P Mid-Cap 400 Value ETF",
    "IJS": "iShares S&P Small-Cap 600 Value ETF",
    "EFV": "iShares MSCI EAFE Value ETF",
    "SPMO": "Invesco S&P 500 Momentum ETF",
    "XMMO": "Invesco S&P MidCap Momentum ETF",
    "XSMO": "Invesco S&P SmallCap Momentum ETF",
    "PDP": "Invesco Dorsey Wright Momentum ETF",
    "IDMO": "Invesco S&P Intl Developed Momentum ETF",
    "RPV": "Invesco S&P 500 Pure Value ETF",
    "SPHD": "Invesco S&P 500 High Dividend Low Volatility ETF",
    "VFMO": "Vanguard U.S. Momentum Factor ETF",
    "VYM": "Vanguard High Dividend Yield ETF",
    "VOE": "Vanguard S&P Mid-Cap Value ETF",
    "VONV": "Vanguard Russell 1000 Value ETF",
    "VOOV": "Vanguard S&P 500 Value ETF",
    "MGV": "Vanguard Mega Cap Value ETF",
    "VBR": "Vanguard Small-Cap Value ETF",
    "VTV": "Vanguard Value ETF",
    "AVLV": "Avantis U.S. Large Cap Value ETF",
    "AVUV": "Avantis U.S. Small Cap Value ETF",
    "JMOM": "JPMorgan U.S. Momentum Factor ETF",
    "GLOV": "Activebeta World Low Vol Plus Equity ETF",
    "DWMF": "WisdomTree International Multifactor ETF",
    "AFLG": "First Trust Active Factor Large Cap Intl ETF",
    "CGDV": "Capital Group Dividend Value ETF",
    "SCHD": "Schwab U.S. Dividend Equity ETF",
    "3070.HK": "Ping An of China CSI HK Dividend ETF"
}
etfs = list(etfs_dict.keys())

In [3]:
# Create an empty DataFrame to store the data of the ETFs
etfs_data = pd.DataFrame(columns=[
    "Symbol", "Establishment Date", "1 Year CAGR (%)", "3 Year CAGR (%)", 
    "5 Year CAGR (%)", "10 Year CAGR (%)", "20 Year CAGR (%)", "Max Period CAGR (%)", 
    "Max Period Annual Volatility (%)", "Max Period Sharpe", 
    "Max Period Sortino", "Max Period MDD (%)", "Max Period Calmar", 
    "5 Year Annual Volatility (%)", "5 Year Sharpe", "5 Year Sortino", 
    "5 Year MDD (%)", "5 Year Calmar", "5 Year Peak Date", 
    "5 Year Recovery Date", "5 Year Recovery Period (days)"
])

# Loop through each ETF in the indices and momentum ETFs
for etf in tqdm(indices + etfs):
    # Get the name of the ETF
    etf_name = {**index_dict, **etfs_dict}.get(etf, etf)
    etfs_data.loc[etf_name, "Symbol"] = etf

    # Get the dataframe for the ETF
    df = get_df(etf, current_date, max_period=True, adj=True)
    
    # Get the establishment date of the ETF
    establishment_date = df.index[0].date()
    etfs_data.loc[etf_name, "Establishment Date"] = establishment_date
    
    # Calculate CAGR values for different periods
    for period, label in zip([252, 252 * 3, 252 * 5, 252 * 10, 252 * 20], 
                             ["1 Year CAGR (%)", "3 Year CAGR (%)", "5 Year CAGR (%)", "10 Year CAGR (%)", "20 Year CAGR (%)"]):
        if len(df) >= period:
            start_price = df["Close"].iloc[- period]
            end_price = df["Close"].iloc[- 1]
            cagr = ((end_price / start_price) ** (1 / (period / 252)) - 1) * 100
            etfs_data.loc[etf_name, label] = cagr

    # Calculate CAGR of maximum period
    start_price = df["Close"].iloc[0]
    end_price = df["Close"].iloc[- 1]
    max_period_cagr = ((end_price / start_price) ** (1 / (len(df) / 252)) - 1) * 100
    etfs_data.loc[etf_name, "Max Period CAGR (%)"] = max_period_cagr

    # Calculate percent change and cumulative return
    df["Percent Change"] = df["Close"].pct_change().dropna()
    df["Cumulative Return"] = (1 + df["Percent Change"]).cumprod()
    daily_returns = df["Percent Change"]
    cumulative_returns = df["Cumulative Return"]
        
    # Calculate annual volatility
    annual_volatility = daily_returns.std() * np.sqrt(252) * 100
    etfs_data.loc[etf_name, "Max Period Annual Volatility (%)"] = annual_volatility

    # Calculate 5 year annual volatility
    if len(df) >= 252 * 5:
        daily_returns_5y = daily_returns.iloc[- 252 * 5:]
        annual_volatility_5y = daily_returns_5y.std() * np.sqrt(252) * 100
        etfs_data.loc[etf_name, "5 Year Annual Volatility (%)"] = annual_volatility_5y

    # Calculate Sharpe ratio
    risk_free_rate = 0.03  # Assuming a risk-free rate of 3%
    sharpe_ratio = (daily_returns.mean() * 252 - risk_free_rate) / (daily_returns.std() * np.sqrt(252))
    etfs_data.loc[etf_name, "Max Period Sharpe"] = sharpe_ratio

    # Calculate 5 year Sharpe ratio
    if len(df) >= 252 * 5:
        sharpe_ratio_5y = (daily_returns_5y.mean() * 252 - risk_free_rate) / (daily_returns_5y.std() * np.sqrt(252))
        etfs_data.loc[etf_name, "5 Year Sharpe"] = sharpe_ratio_5y

    # Calculate Sortino ratio
    negative_returns = daily_returns[daily_returns < 0]
    sortino_ratio = (daily_returns.mean() * 252 - risk_free_rate) / (negative_returns.std() * np.sqrt(252))
    etfs_data.loc[etf_name, "Max Period Sortino"] = sortino_ratio

    # Calculate 5 year Sortino ratio
    if len(df) >= 252 * 5:
        negative_returns_5y = daily_returns_5y[daily_returns_5y < 0]
        sortino_ratio_5y = (daily_returns_5y.mean() * 252 - risk_free_rate) / (negative_returns_5y.std() * np.sqrt(252))
        etfs_data.loc[etf_name, "5 Year Sortino"] = sortino_ratio_5y

    # Calculate maximum drawdown
    df["Peak"] = df["Cumulative Return"].cummax()
    df["Drawdown"] = (df["Cumulative Return"] - df["Peak"]) / df["Peak"]
    drawdowns = df["Drawdown"]
    mdd = drawdowns.min() * 100
    mdd_date = drawdowns.idxmin()
    etfs_data.loc[etf_name, "Max Period MDD (%)"] = mdd

    # Calculate 5 year maximum drawdown
    if len(df) >= 252 * 5:
        drawdowns_5y = drawdowns[- 252 * 5:]
        mdd_5y = drawdowns_5y.min() * 100
        etfs_data.loc[etf_name, "5 Year MDD (%)"] = mdd_5y
        mdd_date_5y = drawdowns_5y.idxmin()

        # Find the peak date before the maximum drawdown
        peak_mask_5y = (df.index <= mdd_date_5y) & (df["Cumulative Return"] == df["Peak"])
        peak_date_5y = df.index[peak_mask_5y][-1]
        etfs_data.loc[etf_name, "5 Year Peak Date"] = peak_date_5y.date()

        # Find the recovery date after the maximum drawdown
        rec_mask_5y = (df.index > mdd_date_5y) & (df["Cumulative Return"] >= df.loc[peak_date_5y, "Cumulative Return"])
        rec_date_5y = df.index[rec_mask_5y][0] if len(df.index[rec_mask_5y]) > 0 else None
        if rec_date_5y is not None:
            etfs_data.loc[etf_name, "5 Year Recovery Date"] = rec_date_5y.date()
            rec_period_5y = (rec_date_5y - peak_date_5y).days
            etfs_data.loc[etf_name, "5 Year Recovery Period (days)"] = rec_period_5y

        # Calculate the two most significant maximum drawdown in the 5 year period
        drawdowns_5y = drawdowns_5y.sort_values()
        mdd_5y = drawdowns_5y.iloc[0] * 100

    # Calculate Calmar ratio
    calmar_ratio = (daily_returns.mean() * 252 - risk_free_rate) / abs(mdd / 100)
    etfs_data.loc[etf_name, "Max Period Calmar"] = calmar_ratio

    # Calculate 5 year Calmar ratio
    if len(df) >= 252 * 5:
        calmar_ratio_5y = (daily_returns_5y.mean() * 252 - risk_free_rate) / abs(mdd_5y / 100)
        etfs_data.loc[etf_name, "5 Year Calmar"] = calmar_ratio_5y

100%|██████████| 36/36 [02:25<00:00,  4.03s/it]


In [4]:
etfs_data

,Symbol,Establishment Date,1 Year CAGR (%),3 Year CAGR (%),5 Year CAGR (%),10 Year CAGR (%),20 Year CAGR (%),Max Period CAGR (%),Max Period Annual Volatility (%),Max Period Sharpe,...,Max Period MDD (%),Max Period Calmar,5 Year Annual Volatility (%),5 Year Sharpe,5 Year Sortino,5 Year MDD (%),5 Year Calmar,5 Year Peak Date,5 Year Recovery Date,5 Year Recovery Period (days)
Hang Seng Index,^HSI,1986-12-31,30.658087,12.281009,1.427137,0.666967,3.423373,6.387816,25.648932,0.254849,...,-65.18186,0.100283,24.992313,0.0722,0.107226,-55.700772,0.032396,2018-01-26,NaN,NaN
S&P 500,^GSPC,1927-12-30,17.624012,22.644069,15.02388,13.567236,8.926321,6.286405,18.95244,0.258394,...,-86.189579,0.056819,17.097581,0.723109,0.996145,-25.425097,0.486268,2022-01-03,2024-01-19,746
NASDAQ Composite,^IXIC,1971-02-05,27.087076,28.495116,15.495108,17.493217,12.630503,10.435165,20.149743,0.444737,...,-77.932386,0.114989,22.743864,0.612548,0.861867,-36.39528,0.382789,2021-11-19,2024-02-29,832
ProShares Ultra QQQ,QLD,2006-06-21,41.257103,54.382476,25.406448,33.41748,NaN,24.7135,44.096462,0.654556,...,-83.128874,0.347215,45.64499,0.654198,0.924908,-63.684482,0.468887,2021-11-19,2024-05-24,917
ProShares UltraPro QQQ,TQQQ,2010-02-11,51.770591,74.994133,27.76844,41.268912,NaN,42.522987,61.20238,0.839753,...,-81.659848,0.629378,67.856221,0.652862,0.921351,-81.659848,0.542504,2021-11-19,2024-12-04,1111
iShares MSCI USA Momentum Factor ETF,MTUM,2013-04-18,27.063052,25.654415,12.987059,15.612172,NaN,15.180472,19.543393,0.668055,...,-34.082265,0.383075,20.796521,0.546339,0.759641,-32.284528,0.351932,2021-11-03,2024-03-04,852
iShares MSCI Intl Momentum Factor ETF,IMTM,2015-01-27,23.931988,24.284873,10.290198,9.441219,NaN,8.437768,20.117614,0.353274,...,-30.683165,0.231627,17.186711,0.489372,0.721548,-30.683165,0.274115,2021-11-08,2024-02-22,836
iShares U.S. Small‑Cap Equity Factor ETF,SMLF,2015-04-30,15.807046,20.128493,16.62777,11.309239,NaN,10.825971,21.761036,0.444181,...,-41.89221,0.230731,21.429236,0.681665,1.068405,-26.277103,0.555905,2024-11-25,2025-08-28,276
iShares S&P U.S. Value ETF,IUSV,2000-08-04,7.986511,18.985651,15.735161,12.218042,8.758023,8.279371,19.176434,0.354601,...,-60.183511,0.112987,15.142907,0.831276,1.215292,-17.951539,0.701218,2022-04-20,2023-02-01,287
iShares S&P Mid-Cap 400 Value ETF,IJJ,2000-07-28,8.965458,15.188169,16.494082,10.748615,8.982778,10.175259,21.923203,0.415337,...,-58.003689,0.156982,20.399987,0.69417,1.093753,-22.679151,0.624409,2024-11-25,NaN,NaN


In [5]:
# Save the data as a CSV file
result_folder = "Result"
filename = os.path.join(result_folder, "etfs_comparison.csv")
etfs_data.to_csv(filename)